# 5G Network Intelligence — External Data Quality Audit

Historical forecasting research. **Default Run All performs no training, calibration or holdout replay.**


## Research question and admission criteria

Can additional public telecom data support transferable load patterns? Admission requires legal access, pinned provenance, unambiguous entity/time keys and continuous observed history. More rows alone are not evidence of better forecasting.


## Environment and saved evidence

Input: completed worker outputs or the repository. Output: paths to saved reports. Missing files stay unavailable; they are never replaced with fabricated results.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Find report folders in the repo and attached Kaggle outputs.
REPORT_DIRS = [Path('reports'), Path('../reports')]  # Repo root or notebooks/ working directory.
if Path('/kaggle/input').exists():
    REPORT_DIRS += sorted({p.parent for p in Path('/kaggle/input').rglob('*validation.csv')})

    REPORT_DIRS += sorted({p.parent for p in Path('/kaggle/input').rglob('*manifest.json')})

def report_path(filename):
    for folder in REPORT_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    return None


## 1. Source register

Keep dataset identities, licenses and domains separate. The commercial counter partitions are not automatically independent networks.


In [ ]:
# Small, explicit metadata view; no raw training data is loaded.
rows = []
for filename in ['external_candidate_manifest.json', 'zenodo_counter_manifest.json']:
    path = report_path(filename)
    if path is None:
        print(f'Unavailable: {filename}')
        continue
    source = json.loads(path.read_text(encoding='utf-8'))
    rows.append({'source': source.get('repo', source.get('doi')), 'revision': source.get('revision', source.get('record')), 'license': source.get('license', 'see pinned LICENSE'), 'files': len(source['files']), 'training_performed': source['training_performed']})
display(pd.DataFrame(rows))


## 2. Schema and duration contradiction

The actual OAI workbook must be checked against its description. A short single-UE series cannot be relabeled as several weeks of hourly beam observations.


In [ ]:
path = report_path('oai_candidate_audit.json')
if path:
    audit = json.loads(path.read_text(encoding='utf-8'))
    display({key: audit[key] for key in ['kpm_records','distinct_ues','span_if_microsecond_time_seconds','median_step_if_microseconds_seconds','hourly_beam_training_eligible','decision']})
else:
    print('OAI workbook audit unavailable.')


## 3. Natural-key conflicts

Duplicate timestamps with different values are not fixed by selecting the first row or averaging silently. Quarantine ambiguous sources.


In [ ]:
path = report_path('zenodo_duplicate_audit.json')
if path:
    display(pd.DataFrame(json.loads(path.read_text(encoding='utf-8'))))
else:
    print('Duplicate audit unavailable.')


## 4. Continuous training prefixes

Aggregation accepts exactly four finite 15-minute observations per hour. Windows never cross gaps. Boundaries are chosen from timestamps before training; later external data is not used by this pilot.


In [ ]:
path = report_path('multisource_preparation.json')
if path:
    preparation = json.loads(path.read_text(encoding='utf-8'))
    display(pd.DataFrame([{'domain': s['source_id'], 'cutoff_utc': s['cutoff_utc'], 'segments': len(s['segments']), 'entity_hours': s['training_rows'], 'future_targets_used': s['future_targets_used']} for s in preparation['sources']]))
else:
    print('Multi-source preparation is not yet attached.')


## Findings and limits

The audit admits sources for research only. It does not prove original-network improvement. Correlated bands must be balanced and cannot be counted as independent test networks. Source code: `scripts/audit_*`, `scripts/prepare_transfer_data.py`; detailed evidence is retained in the JSON reports.
